# 02 · Votação de features

**Features F3, F4 e F6** (critérios 3, 4 e 6) · donos: ver `config/equipe.yaml` · **status: F3, F4 e F6 (registro no MLflow) prontas**

> Cada seção chama o módulo da própria feature (`src/features.py`, `src/votacao.py` e `src/rastreio.py`).

| | |
|---|---|
| **Entradas** | `ciclo_atual.json`, `dev.parquet`, `cfg['selecao']` |
| **Saídas** | `data/processed/features_eleitas.json` · `evidencias/selecao/rankings_<cycle_id>.csv` · `evidencias/votacao/votos_<cycle_id>.csv` · registro no MLflow |

**Regras do enunciado que se aplicam**
- Três métodos no mesmo notebook (SelectKBest, importância do Random Forest e RFE, ou três modelos com feature importance).
- Regra de votação definida **antes** (está em `config.yaml`); registrar tabela de votos, desempate e mínimo de features.
- Ajustar só no treino; com validação cruzada, dentro de cada fold.

Contrato completo na docstring do módulo `src/` correspondente.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto
from src.preprocessamento import carregar_particao, construir_preprocessador, lista_features

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

cycle_id=20261119T182922Z-local | data_version=dv019-aa1a1287


## Seção F3 · três métodos de seleção (`src/features.py`)

In [2]:
from IPython.display import display

from src.features import calcular_rankings_por_fold, listas_aprovadas, salvar_rankings, tabela_rankings

dev = carregar_particao("dev", cfg, RAIZ)
X_bruto = dev[lista_features(cfg)]
y = dev[ctx["colunas"]["alvo"]]
tempo = dev[ctx["colunas"]["tempo"]]
k_top = cfg["selecao"]["k_top"]
print(f"dev: {len(dev):,} linhas | {X_bruto.shape[1]} colunas antes do pré-processador | k_top={k_top}")
print("métodos:", cfg["selecao"]["metodos"])

# Pré-processador e métodos são reajustados só com o treino de cada fold.
rankings, por_fold = calcular_rankings_por_fold(X_bruto, y, tempo, cfg)
tabela = tabela_rankings(rankings, por_fold, k_top)
print(f"{len(tabela)} features após o pré-processador (união dos folds)")

dev: 65,807 linhas | 31 colunas antes do pré-processador | k_top=20
métodos: {'selectkbest': {'score_func': 'mutual_info_classif', 'discretas': ['adults', 'children', 'babies', 'dia_semana_chegada']}, 'random_forest': {'n_estimators': 200, 'max_depth': 12, 'n_jobs': -1}, 'rfe': {'estimador': 'logistic_regression', 'step': 0.1, 'params': {'max_iter': 1000}}}


77 features após o pré-processador (união dos folds)


### Top-k de cada método
Score médio entre os folds. As escalas diferem entre métodos, então só a ordem dentro de cada coluna é comparável. Se houver empate na fronteira, esta tabela corta em ordem alfabética; o conjunto completo entra na concordância abaixo.

### Lista de features aprovadas por método
Cada método aprova as `k_top` melhores. Empatadas na fronteira entram todas, e é por essas listas que a votação conta os votos.

In [3]:
aprovadas = listas_aprovadas(rankings, k_top)
for metodo, lista in aprovadas.items():
    print(f"{metodo} ({len(lista)}): {lista}\n")

selectkbest (20): ['razao_cancel_previo', 'previous_cancellations', 'deposit_type_Non Refund', 'deposit_type_No Deposit', 'lead_time', 'valor_estimado_estadia', 'adr', 'market_segment_Groups', 'agent_1', 'total_of_special_requests', 'required_car_parking_spaces', 'days_in_waiting_list', 'total_noites', 'hotel_City Hotel', 'hotel_Resort Hotel', 'market_segment_Direct', 'distribution_channel_TA/TO', 'stays_in_week_nights', 'distribution_channel_Direct', 'arrival_date_week_number']

random_forest (20): ['razao_cancel_previo', 'deposit_type_No Deposit', 'previous_cancellations', 'deposit_type_Non Refund', 'lead_time', 'total_of_special_requests', 'adr', 'market_segment_Groups', 'valor_estimado_estadia', 'agent_1', 'required_car_parking_spaces', 'arrival_date_week_number', 'customer_type_Transient', 'mes_chegada', 'customer_type_Transient-Party', 'total_noites', 'market_segment_Online TA', 'stays_in_week_nights', 'dia_semana_chegada', 'agent_9']

rfe (21): ['agent_240', 'agent_9', 'deposit_

In [4]:
lado_a_lado = {}
for metodo, score in rankings.items():
    top = score.sort_values(ascending=False, kind="mergesort").head(k_top)
    lado_a_lado[(metodo, "feature")] = top.index.to_list()
    lado_a_lado[(metodo, "score")] = top.round(4).to_list()
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
display(pd.DataFrame(lado_a_lado, index=range(1, len(top) + 1)))

selectkbest                          random_forest                                              rfe        
                        feature   score                        feature   score                                  feature   score
1           razao_cancel_previo  0.1929            razao_cancel_previo  0.1587                                agent_240  1.0000
2        previous_cancellations  0.1830        deposit_type_No Deposit  0.1468                                  agent_9  1.0000
3       deposit_type_Non Refund  0.1762         previous_cancellations  0.1453                  deposit_type_No Deposit  1.0000
4       deposit_type_No Deposit  0.1721        deposit_type_Non Refund  0.1417                  deposit_type_Non Refund  1.0000
5                     lead_time  0.1630                      lead_time  0.0793                      razao_cancel_previo  1.0000
6        valor_estimado_estadia  0.1553      total_of_special_requests  0.0272              required_car_parking_spaces  1.0000
7                           adr  0.1376                            adr  0.0261                                  agent_7  0.9167
8         market_segment_Groups  0.0563          market_segment_Groups  0.0248                   customer_type_Contract  0.9167
9                       agent_1  0.0436         valor_estimado_estadia  0.0218                  customer_type_Transient  0.8944
10    total_of_special_requests  0.0340                        agent_1  0.0178                                  agent_8  0.8889
11  required_car_parking_spaces  0.0297    required_car_parking_spaces  0.0174                                  meal_SC  0.8694
12         days_in_waiting_list  0.0247       arrival_date_week_number  0.0173                                  meal_FB  0.8611
13                 total_noites  0.0216        customer_type_Transient  0.0132  distribution_channel_infrequent_sklearn  0.8056
14             hotel_City Hotel  0.0158                    mes_chegada  0.0123    reserved_room_type_infrequent_sklearn  0.8056
15           hotel_Resort Hotel  0.0158  customer_type_Transient-Party  0.0119                              tem_empresa  0.8056
16        market_segment_Direct  0.0155                   total_noites  0.0089           distribution_channel_Corporate  0.7944
17   distribution_channel_TA/TO  0.0153       market_segment_Online TA  0.0077              distribution_channel_Direct  0.7833
18         stays_in_week_nights  0.0147           stays_in_week_nights  0.0074                         hotel_City Hotel  0.7611
19  distribution_channel_Direct  0.0143             dia_semana_chegada  0.0070                                 agent_28  0.7500
20     arrival_date_week_number  0.0132                        agent_9  0.0070                 market_segment_Corporate  0.7500

### Tabela completa de scores por método
Mesmo índice nos três métodos. `posicao` 1 é a melhor; `folds_presente` diz em quantos folds o one-hot gerou a coluna.

In [5]:
display(tabela.sort_values("posicao_random_forest").round(4))

,score_selectkbest,posicao_selectkbest,aprovada_selectkbest,score_random_forest,posicao_random_forest,aprovada_random_forest,score_rfe,posicao_rfe,aprovada_rfe,folds_presente
feature,,,,,,,,,,
razao_cancel_previo,0.1929,1,1,0.1587,1,1,1.0000,1,1,4
deposit_type_No Deposit,0.1721,4,1,0.1468,2,1,1.0000,1,1,4
previous_cancellations,0.1830,2,1,0.1453,3,1,0.5361,43,0,4
deposit_type_Non Refund,0.1762,3,1,0.1417,4,1,1.0000,1,1,4
lead_time,0.1630,5,1,0.0793,5,1,0.6583,30,0,4
total_of_special_requests,0.0340,10,1,0.0272,6,1,0.6528,32,0,4
adr,0.1376,7,1,0.0261,7,1,0.2444,70,0,4
market_segment_Groups,0.0563,8,1,0.0248,8,1,0.7500,19,1,4
valor_estimado_estadia,0.1553,6,1,0.0218,9,1,0.3528,58,0,4


### Concordância entre os métodos e estabilidade entre folds
Os métodos só agregam na votação se não forem cópias um do outro, e um ranking que muda muito entre folds merece desconfiança. Features empatadas na fronteira entram todas no top-k, que pode ficar maior que k.

In [6]:
tops = {m: set(lista) for m, lista in aprovadas.items()}
print("empates na fronteira (tamanho do top-k):", {m: len(t) for m, t in tops.items() if len(t) > k_top})
display(pd.DataFrame(
    {a: {b: len(tops[a] & tops[b]) for b in tops} for a in tops}
).rename_axis("features em comum no top-k"))

estabilidade = {}
for metodo, grupo in por_fold.groupby("metodo"):
    tops_fold = [set(g.nlargest(k_top, "score", keep="all")["feature"]) for _, g in grupo.groupby("fold")]
    estabilidade[metodo] = {
        f"folds {i + 1}-{i + 2}": len(a & b) / len(a | b) for i, (a, b) in enumerate(zip(tops_fold, tops_fold[1:]))
    }
display(pd.DataFrame(estabilidade).T.round(2).rename_axis("Jaccard do top-k entre folds seguidos"))

em_algum_top = set.union(*tops.values())
parciais = tabela.loc[sorted(em_algum_top), "folds_presente"]
print("no top-k de algum método, mas ausentes em algum fold:", parciais[parciais < cfg["avaliacao"]["cv"]["n_splits"]].to_dict())

empates na fronteira (tamanho do top-k): {'rfe': 21}


,selectkbest,random_forest,rfe
features em comum no top-k,,,
selectkbest,20,14,7
random_forest,14,20,7
rfe,7,7,21


,folds 1-2,folds 2-3,folds 3-4
Jaccard do top-k entre folds seguidos,,,
random_forest,0.82,0.82,0.82
rfe,0.38,0.43,0.48
selectkbest,0.82,0.78,0.86


no top-k de algum método, mas ausentes em algum fold: {'agent_28': 3}


In [7]:
caminhos_rankings = salvar_rankings(tabela, por_fold, ctx, RAIZ)
for nome, caminho in caminhos_rankings.items():
    print(f"{nome}: {caminho.relative_to(RAIZ)}")

rankings: evidencias\selecao\rankings_20261119T182922Z-local.csv
por_fold: evidencias\selecao\rankings_por_fold_20261119T182922Z-local.csv


### Limitações da seleção
- **Cada método tem um viés.** A informação mútua é univariada e ignora redundância. A importância por impureza favorece variáveis contínuas e de muitos valores. O RFE depende dos coeficientes de um modelo linear, e dummies não têm a mesma escala das numéricas padronizadas. É por isso que a votação da F4 combina os três.
- **Atributos redundantes dividem o crédito.** `tem_agente` e `agent_SEM_AGENTE` são exatamente complementares, e `total_noites` é a soma de outras duas colunas. O RFE pode manter uma e descartar a outra sem que a descartada seja inútil.
- **Média entre folds não é seleção por fold.** O treino dos folds finais inclui a validação dos anteriores, então o CV da F5 com um único conjunto eleito sai levemente otimista. Só a validação do último fold e o teste ficam intocados pela seleção. `rankings_por_fold_<cycle_id>.csv` guarda cada fold, caso a F5 queira medir esse efeito.
- **`country` foi retirada antes desta seleção.** O artigo do dataset avisa que a nacionalidade correta costuma ser conhecida só no check-in. Pelo mesmo aviso, `adults`, `children`, `stays_in_*` e `meal` podem carregar parte do problema; ficam como suposição documentada na Etapa 1.

## Seção F4 · votação e eleição (`src/votacao.py`)

**Regra, definida em `config.yaml` antes de rodar:** cada método vota nas features que aprovou, e é eleita a feature com pelo menos 2 dos 3 votos. Dois votos exigem que dois critérios diferentes concordem, sem depender da aprovação unânime de métodos que têm vieses diferentes. Se sobrarem menos de 10 eleitas, o conjunto é completado pelo melhor ranking médio. Empates são desfeitos por mais votos, depois menor ranking médio, depois nome.

In [8]:
from src.votacao import eleger_features, salvar_votacao, votar

regra = cfg["selecao"]["votacao"]
print("regra:", regra, "| k_top:", k_top)

tabela_votos = votar(rankings, cfg)
eleitas = eleger_features(tabela_votos, cfg)
print(f"{len(eleitas)} features eleitas de {len(tabela_votos)}")
print("distribuição de votos:", tabela_votos["votos"].value_counts().sort_index(ascending=False).to_dict())

colunas_voto = [f"voto_{m}" for m in rankings] + ["votos", "ranking_medio"]
display(tabela_votos.assign(eleita=tabela_votos.index.isin(eleitas))[colunas_voto + ["eleita"]].query("votos > 0").round(2))

regra: {'min_votos': 2, 'desempate': 'ranking_medio', 'min_features': 10} | k_top: 20
18 features eleitas de 77
distribuição de votos: {3: 5, 2: 13, 1: 20, 0: 39}


,voto_selectkbest,voto_random_forest,voto_rfe,votos,ranking_medio,eleita
feature,,,,,,
razao_cancel_previo,1,1,1,3,1.00,True
deposit_type_No Deposit,1,1,1,3,2.33,True
deposit_type_Non Refund,1,1,1,3,2.67,True
required_car_parking_spaces,1,1,1,3,7.67,True
market_segment_Groups,1,1,1,3,11.67,True
lead_time,1,1,0,2,13.33,True
previous_cancellations,1,1,0,2,16.00,True
total_of_special_requests,1,1,0,2,16.00,True
hotel_City Hotel,1,0,1,2,18.33,True


In [9]:
n_folds = cfg["avaliacao"]["cv"]["n_splits"]
parciais = tabela.loc[eleitas, "folds_presente"]
print("eleitas ausentes em algum fold (a F5 precisa tratar):", parciais[parciais < n_folds].to_dict())

caminhos_votacao = salvar_votacao(tabela_votos, eleitas, ctx, cfg, RAIZ)
for nome, caminho in caminhos_votacao.items():
    print(f"{nome}: {caminho.relative_to(RAIZ)}")
print("eleitas:", eleitas)

eleitas ausentes em algum fold (a F5 precisa tratar): {}
votos: evidencias\votacao\votos_20261119T182922Z-local.csv
eleitas: data\processed\features_eleitas.json
eleitas: ['razao_cancel_previo', 'deposit_type_No Deposit', 'deposit_type_Non Refund', 'required_car_parking_spaces', 'market_segment_Groups', 'lead_time', 'previous_cancellations', 'total_of_special_requests', 'hotel_City Hotel', 'agent_1', 'total_noites', 'distribution_channel_Direct', 'agent_9', 'arrival_date_week_number', 'valor_estimado_estadia', 'customer_type_Transient', 'adr', 'stays_in_week_nights']


### Limitações da votação
- **Registro no MLflow:** votos, conjunto final, parâmetros e `cycle_id` vão para o MLflow na seção F6, logo abaixo.
- **Votos valem igual:** o RFE é o método menos estável e pesa tanto quanto os outros. Por isso a tabela guarda o ranking médio e os votos de cada método.
- **Um único conjunto por ciclo:** vem da média entre folds, então o CV da F5 tende a sair levemente otimista.

## Seção F6 · registro no MLflow (`src/rastreio.py`)

Um run por ciclo, chamado `02_votacao`, com:
- **tags** `cycle_id` e `data_version` (as mesmas do notebook 01), `etapa`, `origem` e `git_commit`;
- **parâmetros** de toda a seção `selecao` do `config.yaml` (métodos, `k_top`, regra de votação), seed e número de folds;
- **métricas** `n_features_candidatas`, `n_features_eleitas` e quantas features tiveram 0, 1, 2 e 3 votos;
- **artefatos** `selecao/` (rankings), `votacao/` (tabela de votos e `features_eleitas.json`), `features.json` e `contexto/ciclo_atual.json`.

O histórico fica em `mlflow_estado/` (banco SQLite + artefatos). Para abrir a interface: `mlflow ui --backend-store-uri sqlite:///mlflow_estado/mlflow.db`.

In [10]:
import mlflow

from src.rastreio import historico_ciclos, registrar_votacao

run_id = registrar_votacao(ctx, cfg, tabela_votos, eleitas, caminhos_rankings, caminhos_votacao, RAIZ)
print(f"experimento: {cfg['mlflow']['experimento']} | run_id: {run_id}")

# Confere lendo de volta do MLflow, e não da memória do notebook.
salvo = mlflow.get_run(run_id)
assert salvo.data.tags["cycle_id"] == ctx["cycle_id"] and salvo.data.tags["data_version"] == ctx["data_version"]
assert int(salvo.data.metrics["n_features_eleitas"]) == len(eleitas)
print("artefatos:", sorted(a.path for a in mlflow.MlflowClient().list_artifacts(run_id)))

experimento: n3-hotel-cancelamento | run_id: 06391ee83b804ecb9d90ffee8b899e55
artefatos: ['contexto', 'features.json', 'selecao', 'votacao']


### Histórico de ciclos no MLflow
Cada linha é um run. Ciclos anteriores aparecem aqui enquanto `mlflow_estado/` for preservado entre execuções.

In [11]:
historico = historico_ciclos(cfg, RAIZ)
display(historico[["cycle_id", "data_version", "etapa", "run_id", "status", "metrics.n_features_eleitas"]])
print(f"{historico['cycle_id'].nunique()} ciclo(s) e {len(historico)} run(s) no histórico")

,cycle_id,data_version,etapa,run_id,status,metrics.n_features_eleitas
0,20261119T182807Z-local,dv018-f95c3940,02_votacao,0aa32b5f87d044dfaa4fbef64b94fbeb,FINISHED,17.0
1,20261119T182922Z-local,dv019-aa1a1287,02_votacao,06391ee83b804ecb9d90ffee8b899e55,FINISHED,18.0


2 ciclo(s) e 2 run(s) no histórico


### Limitações do registro
- **O histórico só sobrevive se a pasta `mlflow_estado/` for guardada.** No GitHub Actions o runner é apagado no fim; o workflow (F7) precisa recuperá-la no início e guardá-la no fim. Proposta em `docs/f6_mlflow.md`.
- **O MLflow grava o caminho absoluto dos artefatos.** `src/rastreio.py` reescreve esse caminho ao abrir o banco em outra pasta ou máquina, para que o histórico do Actions possa ser aberto localmente.
- **Execuções concorrentes não são suportadas:** o SQLite aceita um escritor por vez. O workflow não deve rodar dois ciclos ao mesmo tempo.